# TC_10: CNN clean baseline
Full diagnostic pipeline on clean image data using simple CNN model.
UQ: Deep Ensemble, MC Dropout
XAI: Grad-CAM

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch


sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_image_data
from src.data_diagnostics.quality_checks import check_image_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.image_models import train_simple_cnn, evaluate_image_model
from src.inference_diagnostics.uncertainty import mc_dropout_prediction_img, deep_ensemble_prediction_img
from src.inference_diagnostics.explainability import gradcam_img, collect_gradcam_feedback
from src.inference_diagnostics.flagging import assign_flags, evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution, plot_gradcam_sample
from src.utils.report_generator import generate_report,save_report
from src.utils.config_loader import load_config, get_image_config, save_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id
from src.inference_diagnostics.explainability import collect_gradcam_feedback_resumable



config = load_config()
tracker = PerformanceTracker()

dataset_name = get_image_config(config)['name']
dataset_short = get_image_config(config)['short_name']
model_name = 'simple_cnn'
test_case = 'TC10'
review_count = config['stage3_inference']['explainability']['gradcam']['max_samples_to_explain']
experiment_id = build_experiment_id(dataset_short, model_name, test_case)
print(f"Experiment: {experiment_id}, reviewing {review_count} samples")

Experiment: fashion_mnist_simple_cnn_TC10, reviewing 100 samples


### Load image data and EDA


In [2]:
train_set, test_set = load_image_data(config)
report = check_image_quality(train_set, config)
plot_class_distribution(report, f"CNN {dataset_name} Baseline", config)

Loading dataset.
Dataset: Fashion-MNIST
Loaded: 60000 train and 10000 test
Image size: 28, Channels: 1
Classes: 10
Class names: ['ankle-boot', 'bag', 'coat', 'dress', 'pullover', 'sandal', 'shirt', 'sneaker', 't-shirt', 'trouser']
EDA started for image data.
Total images: 60000
Classes: 10
Distribution: {0: 6000, 1: 6000, 2: 6000, 3: 6000, 4: 6000, 5: 6000, 6: 6000, 7: 6000, 8: 6000, 9: 6000}
Imbalance ratio: 1.0
EDA completed for Fashion-MNIST
Saved: results/figures\class_distribution_cnn_fashion-mnist_baseline.png


### Train CNN baseline

In [3]:
tracker.start_performance_track()
cnn_model = train_simple_cnn(train_set, config)
tracker.stop_performance_track("CNN training")

tracker.start_performance_track()
cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
tracker.stop_performance_track("CNN evaluation")

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training: Time:703.09s, Memory:701.98MB
 Image model evaluation started.
{'0': {'precision': 0.9624134520276953, 'recall': 0.973, 'f1-score': 0.9676777722526106, 'support': 1000.0}, '1': {'precision': 0.985929648241206, 'recall': 0.981, 'f1-score': 0.9834586466165414, 'support': 1000.0}, '2': {'precision': 0.8791946308724832, 'recall': 0.917, 'f1-score': 0.8976994615761136, 'support': 1000.0}, '3': {'precision': 0.9155339805825242, 'recall': 0.943, 'f1-score': 0.929064039408867, 'support': 1000.0}, '4': {'precision': 0.9218241042345277, 'recall': 0.849, 'f1-score': 0.8839146277980219, 'support': 1000.0}, '5': {'precision': 0.9888663967611336, 'recall': 0.977, 'f1-score': 0.9828973843058351, 'support': 1000.0}, '6': {'precision': 0.7671361502347418, 'recall': 0.817, 'f1-score': 0.7912832929782082, 'support': 1000.0}, '7': {'precision': 0.959122632103689, 'recall': 0.962, 'f1-sc

{'operation': 'CNN evaluation', 'time_seconds': 5.35, 'memory_usage': 15.33}

### Uncertainty Quantification (MC Dropout + Deep Ensemble)

In [4]:
# MC Dropout
tracker.start_performance_track()
mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
tracker.stop_performance_track("CNN MC Dropout")

mc_threshold = round(float(np.percentile(mc_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'mc', mc_threshold)
print(f"MC Dropout uncertainty status:")
print(f"Mean: {mc_uncertainty.mean()}")
print(f"Max: {mc_uncertainty.max()}")
print(f" 90th percentile (threshold): {mc_threshold}")

plot_uncertainty_distribution(mc_uncertainty, "CNN MC Dropout", mc_threshold, config)

MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for images.
CNN MC Dropout: Time:264.14s, Memory:19.04MB
Threshold saved: fashion_mnist_simple_cnn_mc = 0.0265
MC Dropout uncertainty status:
Mean: 0.006260010879486799
Max: 0.11218750476837158
 90th percentile (threshold): 0.0265
Saved: results/figures\uncertainty_distribution_cnn_mc_dropout.png


In [5]:
# Deep Ensemble
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set  )
tracker.stop_performance_track("CNN Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'de', de_threshold)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "CNN Deep Ensemble", de_threshold, config)

Deep Ensemble started for images.
Training model with seed 0
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 1
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 2
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 3
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 4
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Deep Ensemble finished for images.
CNN Deep Ensemble prediction: Time:3539.88s, Memory:3.05MB
Threshold saved: fashion_mnist_simple_cnn_de = 0.0633
Deep Ensembl uncertainty status:
Mean: 0.01210026815533638
Max: 0.16056425869464874
 90th percentile (threshold): 0.0633
Saved: results/figures\unce

### Explainability (Grad-CAM and Human feedback)

In [6]:
tracker.start_performance_track()
heatmaps = gradcam_img(cnn_model, test_set, config)
tracker.stop_performance_track("CNN Grad-CAM")

# Show grid for expert review
plot_gradcam_sample(test_set, heatmaps, review_count, config, f"CNN {dataset_name} Baseline", save=True)


GradCam started.
GradCam progress: 50 / 100 samples.
GradCam progress: 100 / 100 samples.
GradCam finished. Explained: 100 samples.
CNN Grad-CAM: Time:0.54s, Memory:12.05MB
Saved: results/figures\gradcam_sample_cnn_fashion-mnist_baseline.png


### Collect human feedback

In [7]:
# consistency_scores, correct, incorrect, partial = collect_gradcam_feedback(review_count)
# print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")

consistency_scores, correct, incorrect, partial = collect_gradcam_feedback_resumable(
    review_count, config, experiment_id, batch_size=10
)

Scoring: 1 = Correct, 2 = Partial, 3 = Incorrect
Review in batches of 10. Type 'stop' to pause and resume later.

  Saved. Progress: 10/100
  Saved. Progress: 20/100
  Saved. Progress: 30/100
  Saved. Progress: 40/100
  Saved. Progress: 50/100
  Saved. Progress: 60/100
  Saved. Progress: 70/100
  Saved. Progress: 80/100
  Saved. Progress: 90/100
  Only 1, 2 or 3 allowed. Re-enter this batch.
  Saved. Progress: 100/100
Correct: 14, Incorrect: 57, Partial: 29


### Flagging (UQ + Grad-CAM feedback)

In [8]:
# Get true  labels for review_count samples
y_true_reviewed  = np.array([test_set[i][1] for i in range(review_count)])

# MC Dropout
mc_y_pred_reviewed = mc_means_probabilities[:review_count].argmax(axis = 1)
mc_flags_reviewed = assign_flags(mc_uncertainty[:review_count], consistency_scores, mc_threshold, 0.5)
mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(mc_flags_reviewed, "CNN MC + Grad-CAM Reviewed", config)

# Deep Ensemble
de_y_pred_reviewed = de_means_probabilities[:review_count].argmax(axis = 1)
de_flags_reviewed = assign_flags(de_uncertainty[:review_count], consistency_scores, de_threshold, 0.5)
de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(de_flags_reviewed, "CNN DE + Grad-CAM Reviewed", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 57 Accuracy:96.5%
GREEN: Count: 43 Accuracy:100.0%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_+_grad-cam_reviewed.png
RED: Count: 1 Accuracy:0.0%
YELLOW: Count: 58 Accuracy:94.8%
GREEN: Count: 41 Accuracy:100.0%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_+_grad-cam_reviewed.png


### Full flagging with constant consistency for all sample

In [9]:
# Full test set flagging (UQ only, no human review)
fully_y_true = np.array([test_set[i][1] for i in range(len(test_set))])
fake_consistency = np.ones(len(mc_uncertainty))

# MC only
full_mc_y_prediction = mc_means_probabilities.argmax(axis = 1)
mc_flags_full = assign_flags(mc_uncertainty, fake_consistency, mc_threshold, 0.5)
mc_full_result = evaluate_flags(mc_flags_full, full_mc_y_prediction, fully_y_true)
plot_flag_distribution(mc_flags_full, "CNN MC UQ Only Full", config)

# Deep Ensemble only
full_de_y_prediction = de_means_probabilities.argmax(axis = 1)
de_flags_full = assign_flags(de_uncertainty, fake_consistency, de_threshold, 0.5)
de_full_result = evaluate_flags(de_flags_full, full_de_y_prediction, fully_y_true)
plot_flag_distribution(de_flags_full, "CNN DE UQ Only Full", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 999 Accuracy:56.6%
GREEN: Count: 9001 Accuracy:96.4%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_uq_only_full.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 1000 Accuracy:65.3%
GREEN: Count: 9000 Accuracy:97.2%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_uq_only_full.png


In [10]:
# Reviewed-sample flagging (UQ only, no human review)
fake_consistency_reviewed = np.ones(review_count)

# MC only
mc_flags_uq_only = assign_flags(mc_uncertainty[:review_count], fake_consistency_reviewed, mc_threshold, 0.5)
mc_20_result = evaluate_flags(mc_flags_uq_only, mc_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(mc_flags_uq_only, "CNN MC UQ Only reviewed", config)

# Deep Ensemble only
de_flags_uq_only = assign_flags(de_uncertainty[:review_count], fake_consistency_reviewed, de_threshold, 0.5)
de_20_result = evaluate_flags(de_flags_uq_only, de_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(de_flags_uq_only, "CNN DE UQ Only reviewed", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 0 Accuracy:0.0%
GREEN: Count: 100 Accuracy:98.0%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_uq_only_reviewed.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 3 Accuracy:33.3%
GREEN: Count: 97 Accuracy:97.9%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_uq_only_reviewed.png


In [11]:
save_per_sample(
    config,
    experiment_id,
    y_true=y_true_reviewed,
    y_pred=mc_y_pred_reviewed,
    mc_uncertainty=mc_uncertainty[:review_count],
    de_uncertainty=de_uncertainty[:review_count],
    consistency=consistency_scores,
)

Per sample results saved: results/per_sample\fashion_mnist_simple_cnn_TC10.csv (100 rows)


,sample_id,y_true,y_pred,correct,mc_uncertainty,de_uncertainty,consistency
0,0,0,0,1,1.229836e-13,5.092519e-14,0.0
1,1,0,0,1,6.638274e-05,2.973469e-12,1.0
2,2,0,0,1,1.859841e-06,2.227591e-13,0.5
3,3,0,0,1,9.268136e-11,1.669127e-06,1.0
4,4,0,0,1,9.927109e-11,1.980067e-14,0.5
...,...,...,...,...,...,...,...
95,95,0,0,1,2.633126e-18,1.885108e-19,0.0
96,96,0,0,1,3.009599e-08,1.450353e-11,0.5
97,97,0,0,1,2.036836e-05,3.125989e-07,1.0
98,98,0,0,1,5.799682e-10,2.292854e-08,0.0


### Save golden baseline report

In [12]:
cnn_baseline = {
    'model': 'Simple CNN',
    'accuracy': round(cnn_accuracy, 4),
    'classification_report': cnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'flagging_mc_full': mc_full_result,
    'flagging_de_full': de_full_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
}

report_output = generate_report(config, f"{get_image_config(config)['name']} - CNN golden baseline", stage2_result=cnn_baseline)
save_report(report_output, f'{dataset_short.upper()}_TC_10_Simple_CNN_Golden_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.


Per-heatmap std (spatial variance):
  [0] std=0.1343, min=0.0000, max=1.0000
  [1] std=0.1962, min=0.0000, max=1.0000
  [2] std=0.1832, min=0.0000, max=1.0000
  [3] std=0.1954, min=0.0000, max=1.0000
  [4] std=0.1242, min=0.0000, max=1.0000
  [5] std=0.2734, min=0.0000, max=1.0000
  [6] std=0.2095, min=0.0000, max=1.0000
  [7] std=0.2682, min=0.0000, max=1.0000
  [8] std=0.1598, min=0.0000, max=1.0000
  [9] std=0.1778, min=0.0000, max=1.0000
